In [1]:
import random
from datetime import datetime
from datetime import timedelta

from faker import Faker
from pydeequ.analyzers import *
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
spark = SparkSession.builder.appName("Example Deequ").getOrCreate()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
Faker.seed(42)
fake = Faker(['ko_KR', 'en_US'])

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
data = [{
    "id": i + 1,
    "name": fake.name(),
    "age": fake.random_int(min=20, max=65),
    "weight": fake.random_int(min=10, max=250),
    "height": fake.random_int(min=10, max=250),
    "gender": random.choice(['남성', '여성']),
    "address": fake.address(),
    "job": fake.job(),
    "email": fake.email(),
    "signup": (datetime.now() - timedelta(days=random.randint(0, 365))).date().isoformat()
} for i in range(100)]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("weight", IntegerType(), True),
    StructField("height", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("address", StringType(), True),
    StructField("job", StringType(), True),
    StructField("email", StringType(), True),
    StructField("signup", StringType(), True)
])

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
dataframe = spark.createDataFrame(data=data, schema=schema)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
check = (
    Check(spark, CheckLevel.Error, "Basic data checks")
    .hasSize(lambda x: x == 100)
    .isComplete("id")
    .isComplete("name")
    .isComplete("age")
    .isComplete("gender")
    .isComplete("email")
)

result = (VerificationSuite(spark).onData(dataframe).addCheck(check).run())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
analysis_runner = AnalysisRunner(spark)
analysis_result = (
    analysis_runner.onData(dataframe)
    .addAnalyzer(Size())
    .addAnalyzer(Completeness("id"))
    .addAnalyzer(Completeness("name"))
    .addAnalyzer(Completeness("age"))
    .addAnalyzer(Correlation("height", "weight"))
    .addAnalyzer(Completeness("gender"))
    .addAnalyzer(Completeness("address"))
    .addAnalyzer(Completeness("job"))
    .addAnalyzer(Completeness("email"))
    .run())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
result_df = AnalyzerContext.successMetricsAsDataFrame(spark, analysis_result) \
    .withColumn("run_name", lit("daily_batch")) \
    .withColumn("run_id", lit(f"daily_batch_{datetime.now().strftime('%Y%m%d%H%M%S')}")) \
    .withColumn("logical_datetime", lit(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"))
result_df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- entity: string (nullable = true)
 |-- instance: string (nullable = true)
 |-- name: string (nullable = true)
 |-- value: double (nullable = false)
 |-- run_name: string (nullable = false)
 |-- run_id: string (nullable = false)
 |-- logical_datetime: string (nullable = false)

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.